In [1]:
from pathlib import Path

In [2]:
MASK64 = (1 << 64) - 1

MULC = 0x9E3779B97F4A7C15

# x^64 + x^4 + x^3 + x + 1
REDUCE = (1 << 4) | (1 << 3) | (1 << 1) | 1

In [3]:
def rotl(x, n):
    """
    Rotate a 64-bit integer left by n bits.
    """
    return ((x << n) | (x >> (64 - n))) & MASK64


def gf_mul(a, b):
    """
    Multiply in GF(2^64) modulo:

        x^64 + x^4 + x^3 + x + 1
    """

    r = 0

    while b:

        if b & 1:
            r ^= a

        b >>= 1

        a <<= 1

        if a & (1 << 64):
            a = (a & MASK64) ^ REDUCE

    return r & MASK64


def expand(m):
    """
    Spread one byte across 64 bits.
    """
    return (m * 0x0101010101010101) & MASK64


def seal_full(key, msg):
    """
    Return the complete 64-bit internal state.
    """

    s = key & MASK64

    for m in msg:

        s = rotl(s, 11)

        s ^= expand(m)

        s = gf_mul(s, MULC)

    return s


def seal(key, msg):
    """
    Published 24-bit seal:
    top 24 bits of the 64-bit state.
    """
    return seal_full(key, msg) >> 40

In [4]:
file_path = Path("sealed_missions.txt")

missions = []

with file_path.open("r") as f:

    for line in f:

        line = line.strip()

        # Skip empty lines and the comment/header.
        if not line or line.startswith("#"):
            continue

        message_hex, seal_hex = line.split(":")

        message = bytes.fromhex(message_hex)

        observed_seal = int(seal_hex, 16)

        missions.append(
            (message, observed_seal)
        )

In [5]:
print("Number of missions:", len(missions))

assert len(missions) == 5

for i, (message, observed_seal) in enumerate(missions):

    print(
        f"Mission {i + 1}: "
        f"{len(message)} bytes, "
        f"seal = {observed_seal:06x}"
    )

Number of missions: 5
Mission 1: 20 bytes, seal = b9077a
Mission 2: 22 bytes, seal = 501172
Mission 3: 24 bytes, seal = 666b45
Mission 4: 26 bytes, seal = ac89bc
Mission 5: 28 bytes, seal = 91bd1d


In [6]:
for _, observed_seal in missions:

    assert 0 <= observed_seal <= 0xFFFFFF

In [7]:
rows = []
for mission_number, (message, observed_seal) in enumerate(missions, start=1):
    zero_key_output = seal_full(0, message)
    zero_key_seal = zero_key_output >> 40
    target = observed_seal ^ zero_key_seal
    key_bit_effects = []
    for key_bit in range(64):
        basis_key = 1 << key_bit
        basis_output = seal_full(
            basis_key,
            message
        )
        basis_seal = basis_output >> 40
        effect = basis_seal ^ zero_key_seal
        key_bit_effects.append(effect)
    for output_bit in range(24):
        coefficient_mask = 0
        for key_bit in range(64):
            if (key_bit_effects[key_bit] >> output_bit) & 1:
                coefficient_mask |= (
                    1 << key_bit
                )
        rhs = (
            target >> output_bit
        ) & 1
        row = (
            coefficient_mask |
            (rhs << 64)
        )
        rows.append(row)
print()
print("Total equations:", len(rows))
print("Unknown key bits:", 64)


Total equations: 120
Unknown key bits: 64


In [11]:
def solve_gf2(rows, n_variables=64):
    rows = rows.copy()

    pivot_columns = []
    current_row = 0

    # --------------------------------------------------------
    # Gaussian elimination over GF(2)
    # --------------------------------------------------------
    for column in range(n_variables):

        pivot_row = None

        # Find a row containing a 1 in this column
        for r in range(current_row, len(rows)):

            if (rows[r] >> column) & 1:
                pivot_row = r
                break

        # No pivot in this column
        if pivot_row is None:
            continue

        # Move pivot row into position
        rows[current_row], rows[pivot_row] = (
            rows[pivot_row],
            rows[current_row]
        )

        # Eliminate this column from every other row
        for r in range(len(rows)):

            if r == current_row:
                continue

            if (rows[r] >> column) & 1:
                rows[r] ^= rows[current_row]

        pivot_columns.append(column)
        current_row += 1

    # --------------------------------------------------------
    # Check for contradiction: 0 = 1
    # --------------------------------------------------------
    coefficient_mask = (1 << n_variables) - 1

    for r in range(current_row, len(rows)):

        coefficients = rows[r] & coefficient_mask

        rhs = (rows[r] >> n_variables) & 1

        if coefficients == 0 and rhs == 1:
            raise ValueError(
                "Inconsistent GF(2) system."
            )

    # --------------------------------------------------------
    # Recover solution
    # --------------------------------------------------------
    solution = 0

    for row_index, column in enumerate(pivot_columns):

        rhs = (rows[row_index] >> n_variables) & 1

        if rhs:
            solution |= (1 << column)

    return solution, len(pivot_columns)

In [14]:
key, rank = solve_gf2(rows)
print()
print("GF(2) matrix rank:", rank)
if rank == 64:

    print("Full rank: unique 64-bit key determined.")

else:

    print(
        "WARNING: rank is below 64; "
        "the key is not uniquely determined."
    )


GF(2) matrix rank: 64
Full rank: unique 64-bit key determined.


In [15]:
key_hex = f"{key:016x}"

print()
print("Recovered key:")
print(key_hex)


Recovered key:
c7a41f93b65d2e08
